In [1]:
from __future__ import annotations
import os, time, math, sys
os.environ['OMP_NUM_THREADS']='1'
os.environ['MKL_NUM_THREADS']='1'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['VECLIB_MAXIMUM_THREADS']='1'
os.environ['NUMEXPR_NUM_THREADS']='1'
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
_HERE = Path("solver_utils.py").resolve().parent
_SRC = _HERE.parent
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from data.matrix_generation import ConstraintGeneration
from solver_utils import Solver
from constraint_testing.sketch.randomized_svd import RandomizedSVDSolver
from constraint_testing.svd_solver import SVD_Solver
from constraint_testing.linear_conflict_solver import LinearConflictSolver
import scipy.sparse as sp

In [2]:
def get_circular_constraints_example(size=200, inconsistent = False):
    """Generates the 'circular constraints' example matrix and vector."""
    A = []
    a_row = [0] * size
    a_row[0], a_row[1], a_row[2] = 1, -2, 1
    for i in range(size - 2):
        A.append(np.array(a_row))
        a_row = [0] + a_row[:-1]

    A.append(np.array([1] + [0]*(size - 2) + [-1]))
    A.append(np.array([1] + [0]*(size - 1)))
    b = [0]* (size - 2) + [size] + [0]
    if inconsistent:
        A.append(np.array([1] + [0]*(size - 2) + [-1]))
        b.extend([1])

    b = np.array(b)
    A = np.array(A)
    
    A = sp.coo_matrix(A)

    
    return A, b

In [10]:
A, b = get_circular_constraints_example(800, False)
svd_solver = SVD_Solver()
random_svd_solver = RandomizedSVDSolver(target_rank=50, oversample=10, power_iterations=1)
linear_conflict_solver = LinearConflictSolver(solver="lsqr")

In [11]:
qr_start = time.time()
qr_indices, qr_residuals = Solver.analyze_system_with_qr(A, b)
qr_elapsed = time.time() - qr_start
qr_elapsed

0.02969074249267578

In [12]:
rand_svd_start = time.time()
rand_svd_result = random_svd_solver.solve(A, b)
rand_svd_elapsed = time.time() - rand_svd_start
rand_svd_elapsed

0.03542923927307129

In [13]:
qr_indices

array([798, 799, 717, 725, 731, 394, 733, 681, 730, 747, 754, 628, 592,
       659, 718, 759, 311, 401, 427, 363, 791, 735, 784, 732, 692, 675,
       713, 709, 706, 726, 779, 668, 329, 620, 612, 509, 511, 635, 632,
       737, 761, 765, 594, 570, 557, 482, 627, 339, 648, 331, 328, 352,
       337, 341, 697, 373, 399, 550, 767, 601, 414, 410, 408, 357, 440,
       472, 424, 404, 334, 351, 477, 377, 383, 372, 389, 407, 486, 391,
       398, 502, 490, 342, 445, 419, 431, 768, 778, 787, 704, 796, 710,
       719, 715, 727, 724, 785, 773, 698, 699, 690, 689, 702, 701, 691,
       693, 695, 673, 685, 683, 682, 680, 728, 734, 678, 676, 687, 771,
       667, 669, 348, 343, 332, 671, 657, 645, 367, 441, 308, 309, 288,
       333, 481, 485, 499, 498, 622, 609, 611, 631, 629, 528, 541, 536,
       456, 278, 286, 232, 353, 442, 359, 356, 434, 664, 535, 636, 307,
       315, 312, 533, 748, 749, 738, 742, 560, 573, 556, 566, 604, 549,
       546, 524, 589, 579, 595, 593, 527, 362, 762, 375, 756, 74

In [14]:
svd_start = time.time()
svd_result = svd_solver.solve(A, b)
svd_elapsed = time.time() - svd_start
svd_elapsed

0.6302244663238525

In [15]:
linear_start = time.time()
linear_result = linear_conflict_solver.analyze(A, b, verify_iis=True)
linear_elapsed = time.time() - linear_start
linear_elapsed

10.991463899612427

In [16]:
print("QR runtime (s)", qr_elapsed)
print("Dense SVD runtime (s)", svd_elapsed)
print("Randomized SVD runtime (s)", rand_svd_elapsed)
print("Linear conflict solver runtime (s)", linear_elapsed)
print("QR top residual rows", qr_indices[:10])
print("Dense SVD inconsistent rows", svd_result.inconsistent_rows)
print("Randomized SVD inconsistent rows", rand_svd_result.inconsistent_rows)
print("Linear conflict solver conflict rows", linear_result.conflicting_indices)
print("Linear conflict solver IIS verified", linear_result.iis_verified)

QR runtime (s) 0.02969074249267578
Dense SVD runtime (s) 0.6302244663238525
Randomized SVD runtime (s) 0.03542923927307129
Linear conflict solver runtime (s) 10.991463899612427
QR top residual rows [798 799 717 725 731 394 733 681 730 747]
Dense SVD inconsistent rows (398, 399)
Randomized SVD inconsistent rows ()
Linear conflict solver conflict rows (473, 474, 472, 475, 476, 469, 477, 471, 468, 467, 479, 470, 478, 481, 466, 482, 457, 480, 465, 497, 458, 498, 499, 459, 496, 483, 460, 462, 495, 463, 456, 461, 517, 464, 501, 484, 485, 494, 500, 493, 518, 522, 521, 523, 516, 455, 519, 503, 515, 513, 502, 520, 454, 505, 486, 524, 492, 514, 525, 491, 504, 453, 506, 507, 511, 512, 489, 508, 490, 487, 488, 510, 526, 452, 509, 527, 450, 451, 449, 448, 445, 447, 446, 528, 443, 444, 442, 441, 529, 440, 438, 530, 439, 428, 429, 531, 431, 425, 437, 423, 430, 426, 432, 427, 424, 419, 422, 532, 433, 436, 421, 420, 435, 418, 434, 533, 534, 535, 417, 416, 415, 536, 414, 400, 404, 409, 403, 405, 401, 40

In [17]:
svd_residual = b - A @ svd_result.solution
linear_residual = linear_result.residual
qr_residual_norm = np.linalg.norm(qr_residuals)
svd_residual_norm = np.linalg.norm(svd_residual)
rand_svd_residual_norm = np.linalg.norm(rand_svd_result.residual)
linear_residual_norm = np.linalg.norm(linear_residual)
svd_vs_linear = np.linalg.norm(svd_residual - linear_residual)
qr_residual_norm, svd_residual_norm, rand_svd_residual_norm, linear_residual_norm, svd_vs_linear

(np.float64(5.113897837637326e-12),
 np.float64(3.416271044991565e-11),
 np.float64(800.142195937961),
 np.float64(0.09911219721945384),
 np.float64(0.09911219721957533))

In [18]:
list(rand_svd_result.solved_variables.items())[:10]

[]

In [19]:
linear_result.conflicting_indices, linear_result.iis_verified

((473,
  474,
  472,
  475,
  476,
  469,
  477,
  471,
  468,
  467,
  479,
  470,
  478,
  481,
  466,
  482,
  457,
  480,
  465,
  497,
  458,
  498,
  499,
  459,
  496,
  483,
  460,
  462,
  495,
  463,
  456,
  461,
  517,
  464,
  501,
  484,
  485,
  494,
  500,
  493,
  518,
  522,
  521,
  523,
  516,
  455,
  519,
  503,
  515,
  513,
  502,
  520,
  454,
  505,
  486,
  524,
  492,
  514,
  525,
  491,
  504,
  453,
  506,
  507,
  511,
  512,
  489,
  508,
  490,
  487,
  488,
  510,
  526,
  452,
  509,
  527,
  450,
  451,
  449,
  448,
  445,
  447,
  446,
  528,
  443,
  444,
  442,
  441,
  529,
  440,
  438,
  530,
  439,
  428,
  429,
  531,
  431,
  425,
  437,
  423,
  430,
  426,
  432,
  427,
  424,
  419,
  422,
  532,
  433,
  436,
  421,
  420,
  435,
  418,
  434,
  533,
  534,
  535,
  417,
  416,
  415,
  536,
  414,
  400,
  404,
  409,
  403,
  405,
  401,
  402,
  412,
  413,
  407,
  406,
  408,
  411,
  399,
  410,
  537,
  538,
  539,
  398,
  540,

In [20]:
A = np.array([
    [1, 0],  # x = 1
    [0, 1],  # y = 1
    [1, 1],  # x + y = 2
    [1, 0]   # x = 3
])

# Right-hand side vector b
b = np.array([
    1,
    1,
    2,
    3
])

x,y= Solver.analyze_system_with_qr(A, b, verbose = False)
print(x,y)

[3 0 1 2] [-0.8  0.4 -0.4  1.2]


In [21]:
linear_start = time.time()
linear_result = linear_conflict_solver.analyze(A, b)
linear_elapsed = time.time() - linear_start
linear_elapsed

linear_result

LinearConflictResult(is_consistent=False, iterate=array([1.8, 0.6]), residual=array([ 0.8, -0.4,  0.4, -1.2]), residual_norm=1.5491933384829668, iterations=2, conflict_analysis=ConflictAnalysis(residual_norm=1.5491933384829668, conflict_indices=(3, 0, 1, 2), threshold=1e-08), conflicting_indices=(3, 0, 1, 2), iis_verified=False)

In [40]:
svd_start = time.time()
svd_result = svd_solver.solve(A, b)
svd_elapsed = time.time() - svd_start
svd_elapsed

svd_result

SVDResult(solution=array([1.8, 0.6]), solved_variables={0: 2.0, 1: 1.0}, inconsistent_rows=(0, 2, 3), reduced_matrix=array([], shape=(0, 0), dtype=float64), reduced_rhs=array([], dtype=float64), rank=2, singular_values=array([1.90211303, 1.1755705 ]))

In [2]:
linear_start = time.time()
linear_result = linear_conflict_solver.analyze(A, b, verify_iis=True)
linear_elapsed = time.time() - linear_start
linear_elapsed

linear_result

------------------
R diagonal
[[-7.48331477 -4.81070235]
 [ 0.          5.5549206 ]
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]]
------------------
c matrix
[-6.61471574  1.56232142  0.05233974  1.02564519 -1.41421356]
------------------
[0.703125 0.28125 ]
[0.703125 0.28125 ]
[ 0.046875  0.234375  0.296875 -1.703125  0.09375 ]


In [3]:
print(A)

[3 2 1 4 0]


In [10]:

s = time.time()
A,b,c = ConstraintGeneration.create_random_sparse_constraints(50, 50, True, 0)
e = time.time()


In [5]:
s = time.time()
x = Solver.analyze_system_with_qr(A,b)
e = time.time()

print(e - s)

[ 0.00000000e+00 -3.33066907e-16  1.11022302e-16  0.00000000e+00
  1.11022302e-16]
0.0035305023193359375


LinAlgError: Singular matrix

In [27]:
print(sorted(x[1]))

[np.float64(-0.49999999999999967), np.float64(-0.21858304277948803), np.float64(-0.08536330219050325), np.float64(-0.006483629773681332), np.float64(-1.5253942553528077e-10), np.float64(-1.0128387017971363e-10), np.float64(-7.429534765179824e-11), np.float64(-7.286771186443275e-11), np.float64(-6.216727133079303e-11), np.float64(-5.83477710591751e-11), np.float64(-4.9855009010002505e-11), np.float64(-2.874667170971179e-11), np.float64(-2.340394544830815e-11), np.float64(-2.155309264395555e-11), np.float64(-1.9207746504434908e-11), np.float64(-1.4787615576494773e-11), np.float64(-1.0193068611386025e-11), np.float64(-9.230172182128626e-12), np.float64(-8.305911514128184e-12), np.float64(-7.644329613754053e-12), np.float64(-6.9714234385287455e-12), np.float64(-6.627476345499872e-12), np.float64(-6.288525256081812e-12), np.float64(-6.2156946256664014e-12), np.float64(-5.355382803884368e-12), np.float64(-4.657385588302532e-12), np.float64(-3.5571545708990016e-12), np.float64(-3.541167359344

In [28]:
import numpy as np
import scipy.linalg as la

Q, R, P = la.qr(A.toarray(), pivoting=True)

In [9]:
print(np.diag(Q))

[-7.07106781e-01 -7.07106781e-01  0.00000000e+00  0.00000000e+00
  0.00000000e+00 -7.85046229e-17 -5.00000000e-01 -3.03381438e-18
 -3.87298335e-01  3.16227766e-01]


In [ ]:
print(x)

In [ ]:
A1 = sp.csc_matrix([[1, 0,0], [0, 1,0],[0,0,1]])
b1 = np.array([10, 20,30])
print(Solver.is_conflicting_using_QR(A1, b1))




In [ ]:

A4 = sp.csc_matrix([[1,0,0,-1],[0,1, 1,0], [0,2, 2,0], [0,0,0,1]]) # Rank 1
b4 = np.array([12,10, 30,1])
A,B = Solver.analyze_system_with_qr(A4, b4)

print(A,B)

In [ ]:
Solver.is_conflicting_using_QR(A4, b4)